本Notebook将使用MedSAM（Medical Segment Anything Model）的原生模型，在多个公开的医学图像分割数据集上进行基线性能评估。我们将涵盖MRI、CT和超声图像等不同模态的数据集，每个步骤均提供清晰的注释和结果的可视化展示，包括分割效果图和评价指标表格。请确保在运行之前将Colab的运行时类型设置为GPU（Runtime > Change runtime type > GPU），以加速模型推理过程。

环境配置与MedSAM模型准备

首先，安装必要的库并获取MedSAM模型代码和预训练权重。
安装依赖库：包括AWS CLI（用于下载数据）、Nibabel（用于读取医学影像格式）、OpenCV（用于图像读取）以及MedPy（用于评价指标计算）。
克隆MedSAM仓库：从官方GitHub获取MedSAM的代码实现。
安装MedSAM：安装MedSAM依赖并设置环境。
下载MedSAM预训练权重：使用wget从官方发布的权重地址下载MedSAM的ViT-Base模型权重文件，并放置在合适路径。

In [ ]:
!pip install -q awscli nibabel medpy opencv-python
!git clone https://github.com/bowang-lab/MedSAM.git
%cd MedSAM
!pip install -e .
%cd ..
# 下载MedSAM预训练模型权重并保存为 medsam_vit_b.pth
!wget -O medsam_vit_b.pth "https://huggingface.co/flaviagiammarino/medsam-vit-base/resolve/main/pytorch_model.bin"


接下来，我们加载MedSAM模型以准备进行推理。

In [ ]:
import torch
from segment_anything import sam_model_registry, SamPredictor

# 加载MedSAM模型（ViT-Base版本）和预测器
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_b"](checkpoint="medsam_vit_b.pth").to(device)
predictor = SamPredictor(sam)


数据集下载与预处理

本节我们将下载三个不同模态的医学图像分割数据集，并进行必要的预处理。所选数据集包括：
MRI: 选择Medical Segmentation Decathlon中的Hippocampus数据集（脑MRI的海马结构分割）
academictorrents.com
academictorrents.com
。
CT: 选择Medical Segmentation Decathlon中的Spleen数据集（腹部CT的脾脏分割）
academictorrents.com
academictorrents.com
。
超声: 选择乳腺超声图像分割数据集BUSB (Breast Ultrasound Images Dataset, BUSI)，包含乳腺良/恶性肿块及相应分割
academictorrents.com
academictorrents.com
。
每个数据集下载后都会进行解压和基本处理，以方便后续推理和评估。
1. 下载MRI数据集（Hippocampus）
Hippocampus数据集来自Medical Segmentation Decathlon Task04
academictorrents.com
。该数据集包含T2 MRI的体积，标注了左右海马的分割(mask标签值1和2分别表示两个结构)。 我们使用AWS开放数据接口下载Task04_Hippocampus数据集的压缩包，并解压以获取图像和标签：

In [ ]:
# 使用AWS CLI无认证下载Decathlon Task04 (Hippocampus) 数据集 (~29MB)
!aws s3 cp --no-sign-request s3://msd-for-monai/Task04_Hippocampus.tar .
!tar -xf Task04_Hippocampus.tar
!rm Task04_Hippocampus.tar  # 解压后删除压缩包节省空间

# 列出文件结构（截取部分）
!find Task04_Hippocampus -maxdepth 2 -type d -printf '%P\n'


接下来读取MRI体积数据和对应的标签。在评估中，我们将使用训练集作为测试图像（MedSAM并未在该特定数据上训练过，因此训练集可以用作评估基线）。我们将利用Nibabel库读取NIfTI格式的体数据，并提取其中的2D切片进行推理：

In [ ]:
import nibabel as nib
import numpy as np

# 获取Hippocampus训练集文件列表
image_files = sorted([f for f in !ls Task04_Hippocampus/imagesTr])
label_files = sorted([f for f in !ls Task04_Hippocampus/labelsTr])

print(f"共有 {len(image_files)} 副MRI体积用于评估.")

# 准备存储MRI数据集的测试切片和标签
mri_slices = []      # 将保存二维切片图像 (numpy数组)
mri_slice_masks = [] # 将保存对应的二维GT掩码
mri_slice_labels = []# 保存掩码对应的解剖结构标签（1=左海马, 2=右海马）

for img_file, lbl_file in zip(image_files, label_files):
    img_path = f"Task04_Hippocampus/imagesTr/{img_file}"
    lbl_path = f"Task04_Hippocampus/labelsTr/{lbl_file}"
    # 载入NIfTI体数据
    img_nii = nib.load(img_path)
    lbl_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata()        # 图像体数据，形状 (Z, Y, X)
    lbl_data = lbl_nii.get_fdata().astype(np.uint8)  # 标签体数据，形状同上
    
    # 遍历每个切片（以轴0为切片方向，即轴向切片）
    num_slices = img_data.shape[0]
    for k in range(num_slices):
        slice_img = img_data[k, :, :]
        slice_lbl = lbl_data[k, :, :]
        # 如果该切片存在任何一个海马结构的标注，则纳入评估
        if np.any(slice_lbl > 0):
            # 图像切片灰度值缩放到0-255区间，并转换为3通道uint8
            mn, mx = slice_img.min(), slice_img.max()
            slice_img_norm = ((slice_img - mn) / (mx - mn + 1e-8) * 255.0).astype(np.uint8)
            slice_img_rgb = np.stack([slice_img_norm]*3, axis=-1)  # (H,W,3)
            
            # 将切片及对应标签加入列表
            mri_slices.append(slice_img_rgb)
            mri_slice_masks.append(slice_lbl)  # 多类别标签（值0,1,2）
            mri_slice_labels.append(lbl_file)  # 记录该切片所属体数据文件


以上代码遍历了MRI体数据的每个切片，仅保留含有海马标注的切片，以减少不必要的计算。每个切片被归一化为三通道8位图像（MedSAM需要RGB图像输入）。mri_slices列表将用于模型推理，mri_slice_masks为对应的真值掩码。
2. 下载CT数据集（Spleen）
Spleen数据集来自Medical Segmentation Decathlon Task09
academictorrents.com
。该数据集提供腹部CT体数据及脾脏的分割标签（二值掩码，1表示脾脏）。我们同样使用AWS接口下载Task09_Spleen数据集，并解压：

In [ ]:
# 下载Decathlon Task09 (Spleen) 数据集 (~1.5GB)
!aws s3 cp --no-sign-request s3://msd-for-monai/Task09_Spleen.tar .
!tar -xf Task09_Spleen.tar
!rm Task09_Spleen.tar

# 列出Spleen数据集目录结构
!find Task09_Spleen -maxdepth 1 -type d -printf '%P\n'


下载完成后，我们读取CT体数据和标签，并提取有脾脏的切片进行评估：

In [ ]:
# 获取Spleen数据集文件列表
ct_image_files = sorted([f for f in !ls Task09_Spleen/imagesTr])
ct_label_files = sorted([f for f in !ls Task09_Spleen/labelsTr])
print(f"共有 {len(ct_image_files)} 副CT体积用于评估.")

ct_slices = []
ct_slice_masks = []

for img_file, lbl_file in zip(ct_image_files, ct_label_files):
    img_path = f"Task09_Spleen/imagesTr/{img_file}"
    lbl_path = f"Task09_Spleen/labelsTr/{lbl_file}"
    img_nii = nib.load(img_path)
    lbl_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata()
    lbl_data = lbl_nii.get_fdata().astype(np.uint8)
    # 遍历轴向切片
    for k in range(img_data.shape[0]):
        slice_img = img_data[k, :, :]
        slice_lbl = lbl_data[k, :, :]
        if np.any(slice_lbl == 1):  # 若该切片含有脾脏
            # 将CT切片灰度值裁剪到 [-1000, 1000] HU 范围，并归一化到0-255
            slice_clip = np.clip(slice_img, -1000, 1000)
            mn, mx = slice_clip.min(), slice_clip.max()
            slice_img_norm = ((slice_clip - mn) / (mx - mn + 1e-8) * 255.0).astype(np.uint8)
            slice_img_rgb = np.stack([slice_img_norm]*3, axis=-1)
            ct_slices.append(slice_img_rgb)
            ct_slice_masks.append(slice_lbl)  # 二值掩码（0背景，1脾脏）


在以上代码中，对于CT强度值，我们对每个切片裁剪在[-1000,1000]范围以排除空气和高密度异常值，然后进行min-max归一化。这样可以保留软组织和脏器的对比。只提取包含脾脏的切片到ct_slices列表中。
3. 下载超声数据集（BUSB/BUSI）
BUSB（Breast Ultrasound Images Dataset）数据集包含780张超声图像及其分割掩码，包括正常、良性和恶性三类病例
academictorrents.com
datasetninja.com
。其中正常类没有肿块（无分割掩码），良性和恶性病例有肿瘤掩码。我们下载该数据集的公开发布版本，并解压得到图像及标注。这里我们假设数据集以zip文件提供，并采用bash命令获取：

In [ ]:
# 下载乳腺超声图像数据集 (约250MB)
!wget -q -O BUSI.zip "https://academictorrents.com/download/d0b7b7ae40610bbeaea385aeb51658f527c86a16.torrent?torrent" || echo "Download started"
!unzip -q BUSI.zip -d BUSI_Dataset
!rm BUSI.zip


📓 说明: 上述命令使用Academic Torrents获取数据集。如果下载缓慢或失败，建议用户手动下载数据集并上传至Colab环境。
解压后，数据集通常包含三个子文件夹：benign/, malignant/, normal/，每个文件夹下有图像和对应的掩码文件。掩码文件命名为原图文件名加“_mask”后缀（若一个图像有多个肿块，则会有“_mask_2”等多份掩码）
stackoverflow.com
stackoverflow.com
。我们读取良性和恶性文件夹下的图像和掩码：

In [ ]:
import cv2
import os
import glob

ultrasound_images = []
ultrasound_masks = []

# 处理良性和恶性文件夹
for cls in ["benign", "malignant"]:
    image_paths = glob.glob(f"BUSI_Dataset/{cls}/*.png")
    for img_path in image_paths:
        if "_mask" in img_path:
            continue  # 跳过掩码文件
        # 读取超声图像 (灰度PNG, cv2.imread默认读取为BGR三通道)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # 构造与图像尺寸相同的空白掩码
        mask_total = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
        # 图像文件名基础部分（去掉路径和扩展名）
        base_name = os.path.splitext(img_path)[0]
        # 合并该图像的所有掩码文件（可能有多个肿块）
        mask_idx = 1
        while True:
            mask_path = f"{base_name}_mask.png" if mask_idx == 1 else f"{base_name}_mask_{mask_idx}.png"
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is not None:
                    mask_binary = (mask > 127).astype(np.uint8)
                    mask_total = np.logical_or(mask_total, mask_binary).astype(np.uint8)
                mask_idx += 1
            else:
                break
        # 如果该图像存在肿块标注，则保存
        if mask_total.sum() > 0:
            ultrasound_images.append(img_rgb)
            ultrasound_masks.append(mask_total)
            
print(f"乳腺超声图像总数: {len(ultrasound_images)} (良性+恶性), 掩码总数: {len(ultrasound_masks)}")


In [ ]:
import math
from scipy.ndimage import distance_transform_edt

def compute_dice(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """计算Dice系数"""
    inter = np.logical_and(pred_mask, true_mask).sum()
    size_pred = pred_mask.sum()
    size_true = true_mask.sum()
    if size_pred + size_true == 0:
        return 1.0  # 都为空，视为完全重叠
    return 2.0 * inter / (size_pred + size_true)

def compute_iou(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """计算IoU"""
    inter = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()
    if union == 0:
        return 1.0
    return inter / union

def compute_hausdorff(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """计算Hausdorff距离（基于欧几里得距离的最大值）"""
    # 若其中一个为空集，则返回极大值（此处返回Infinity表示无法计算）
    if pred_mask.sum() == 0 or true_mask.sum() == 0:
        return math.inf
    # 计算真值掩码的距离变换（背景像素到最近真值前景的距离）
    dt_true = distance_transform_edt(~true_mask.astype(bool))
    # 计算预测掩码的距离变换
    dt_pred = distance_transform_edt(~pred_mask.astype(bool))
    # 真值边界到预测区域的最大距离
    hd1 = dt_pred[true_mask.astype(bool)].max()
    # 预测边界到真值区域的最大距离
    hd2 = dt_true[pred_mask.astype(bool)].max()
    return max(hd1, hd2)


以上函数中使用scipy.ndimage.distance_transform_edt来高效计算距离变换，从而获得Hausdorff距离。 接下来，我们对各数据集分别进行推理和评估。在每次推理之前，我们使用SamPredictor.set_image设定当前图像，这会计算图像的特征嵌入。然后对于需要分割的目标，我们提供边界框提示给模型以获得掩码预测
huggingface.co
。
评估MRI数据集（脑海马）
对于MRI的海马数据集，每张切片可能包含左、右两个结构，我们将分别进行预测：
如果切片中存在左海马（标签值1）或右海马（标签值2），我们分别计算各自的边界框提示给MedSAM。
将模型预测的两个掩码合并为整个切片的预测。
计算该切片左、右海马的Dice、IoU和Hausdorff距离。对于Dice和IoU，我们对左右两侧分别计算，然后求平均作为该切片总体指标；Hausdorff距离我们取左右结构距离的最大值作为切片结果（因为Hausdorff通常衡量整体segmentation的最差误差）。


In [ ]:
# 评估MRI (Hippocampus) 数据集
dice_list_hc = []
iou_list_hc = []
hd_list_hc = []

for img_rgb, true_mask in zip(mri_slices, mri_slice_masks):
    predictor.set_image(img_rgb)  # 计算图像嵌入
    pred_mask_total = np.zeros(true_mask.shape, dtype=bool)
    # 对每个结构分别预测 (标签1和2分别对应两个海马)
    for label_val in [1, 2]:
        # 跳过不存在的结构
        if np.sum(true_mask == label_val) == 0:
            continue
        # 计算该结构的边界框
        ys, xs = np.where(true_mask == label_val)
        y_min, y_max = ys.min(), ys.max()
        x_min, x_max = xs.min(), xs.max()
        input_box = np.array([x_min, y_min, x_max, y_max])
        # 使用边界框提示进行预测
        masks, scores, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
        pred_mask = masks[0]  # 输出掩码
        # 将该结构的预测掩码累加到总掩码
        pred_mask_total = np.logical_or(pred_mask_total, pred_mask)
    # 计算评价指标
    dice = compute_dice(pred_mask_total, true_mask > 0)
    iou = compute_iou(pred_mask_total, true_mask > 0)
    hd = compute_hausdorff(pred_mask_total, true_mask > 0)
    dice_list_hc.append(dice)
    iou_list_hc.append(iou)
    hd_list_hc.append(hd)

# 计算平均指标
dice_mean_hc = np.mean(dice_list_hc)
iou_mean_hc = np.mean(iou_list_hc)
hd_mean_hc = np.mean([d for d in hd_list_hc if math.isfinite(d)])
print(f"Hippocampus MRI数据集: 平均Dice = {dice_mean_hc:.4f}, 平均IoU = {iou_mean_hc:.4f}, 平均Hausdorff距离 = {hd_mean_hc:.2f} pixel")


在上述代码中，我们将左右海马的预测结果合并后，与整个真值掩码进行比较，从而得到每张切片的Dice和IoU（左右结构共同评估）以及Hausdorff距离。最后计算了所有切片的平均值作为该数据集的整体性能指标。 让我们从该数据集中可视化一个示例切片的分割结果：

In [ ]:
import matplotlib.pyplot as plt

# 随机选择一个示例切片来可视化
idx = np.random.randint(0, len(mri_slices))
test_img = mri_slices[idx]
test_true_mask = mri_slice_masks[idx]

# 使用MedSAM对该切片进行分割（类似上面流程）
predictor.set_image(test_img)
pred_mask_total = np.zeros(test_true_mask.shape, dtype=bool)
for label_val in [1, 2]:
    if np.sum(test_true_mask == label_val) == 0:
        continue
    ys, xs = np.where(test_true_mask == label_val)
    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()
    input_box = np.array([x_min, y_min, x_max, y_max])
    masks, _, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
    pred_mask_total = np.logical_or(pred_mask_total, masks[0])

# 绘制原始图像，真值和预测结果叠加
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(test_img[...,0], cmap='gray')
axes[0].imshow(np.ma.masked_where(test_true_mask==0, test_true_mask), cmap='autumn', alpha=0.5)
axes[0].set_title("真值分割 (海马)")
axes[0].axis('off')

axes[1].imshow(test_img[...,0], cmap='gray')
axes[1].imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total), cmap='autumn', alpha=0.5)
axes[1].set_title("MedSAM预测分割")
axes[1].axis('off')
plt.show()


上述可视化左侧显示了MRI切片上真值的海马分割区域（橙色半透明覆盖），右侧为MedSAM模型的预测结果。通过肉眼可观察预测与真值的重合情况。
评估CT数据集（脾脏）
对于CT脾脏数据集，每张切片至多包含一个目标（脾脏），因此我们直接对每个切片：
计算脾脏区域的边界框，使用MedSAM预测掩码。
计算Dice、IoU和Hausdorff距离。

In [ ]:
dice_list_spleen = []
iou_list_spleen = []
hd_list_spleen = []

for img_rgb, true_mask in zip(ct_slices, ct_slice_masks):
    predictor.set_image(img_rgb)
    # 计算真值掩码的边界框（脾脏标注值为1）
    ys, xs = np.where(true_mask == 1)
    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()
    input_box = np.array([x_min, y_min, x_max, y_max])
    # 使用边界框提示预测
    masks, _, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
    pred_mask = masks[0]
    # 计算指标
    dice_list_spleen.append(compute_dice(pred_mask, true_mask == 1))
    iou_list_spleen.append(compute_iou(pred_mask, true_mask == 1))
    hd_list_spleen.append(compute_hausdorff(pred_mask, true_mask == 1))

dice_mean_spleen = np.mean(dice_list_spleen)
iou_mean_spleen = np.mean(iou_list_spleen)
hd_mean_spleen = np.mean([d for d in hd_list_spleen if math.isfinite(d)])
print(f"Spleen CT数据集: 平均Dice = {dice_mean_spleen:.4f}, 平均IoU = {iou_mean_spleen:.4f}, 平均Hausdorff距离 = {hd_mean_spleen:.2f} pixel")


同样，我们挑选一个CT切片的结果进行可视化：

In [ ]:
# 随机选择一个CT切片进行可视化
idx = np.random.randint(0, len(ct_slices))
test_img = ct_slices[idx]
test_mask = ct_slice_masks[idx]

# 使用MedSAM预测该切片
predictor.set_image(test_img)
ys, xs = np.where(test_mask == 1)
y_min, y_max = ys.min(), ys.max()
x_min, x_max = xs.min(), xs.max()
input_box = np.array([x_min, y_min, x_max, y_max])
masks, _, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
pred_mask = masks[0]

# 可视化结果
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(test_img[...,0], cmap='gray')
axes[0].imshow(np.ma.masked_where(test_mask==0, test_mask), cmap='spring', alpha=0.5)
axes[0].set_title("真值分割 (脾脏)")
axes[0].axis('off')

axes[1].imshow(test_img[...,0], cmap='gray')
axes[1].imshow(np.ma.masked_where(pred_mask==0, pred_mask), cmap='spring', alpha=0.5)
axes[1].set_title("MedSAM预测分割")
axes[1].axis('off')
plt.show()


左图为CT切片上的真值脾脏区域（绿色高亮），右图为MedSAM预测的分割。可以看到MedSAM在大多数切片上能够勾画出脾脏的大致轮廓，但在边界细节上可能存在误差。
评估超声数据集（乳腺肿块）
对乳腺超声图像，我们同样逐张评估：
每张图像可能包含一个或多个肿块（我们已将所有肿块掩码合并）。
计算整个肿块区域的边界框进行预测（相当于对图中所有病灶统一分割）。
计算评价指标。

In [ ]:
dice_list_us = []
iou_list_us = []
hd_list_us = []

for img_rgb, true_mask in zip(ultrasound_images, ultrasound_masks):
    predictor.set_image(img_rgb)
    ys, xs = np.where(true_mask == 1)
    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()
    input_box = np.array([x_min, y_min, x_max, y_max])
    masks, _, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
    pred_mask = masks[0]
    # 评价指标
    dice_list_us.append(compute_dice(pred_mask, true_mask == 1))
    iou_list_us.append(compute_iou(pred_mask, true_mask == 1))
    hd_list_us.append(compute_hausdorff(pred_mask, true_mask == 1))

dice_mean_us = np.mean(dice_list_us)
iou_mean_us = np.mean(iou_list_us)
hd_mean_us = np.mean([d for d in hd_list_us if math.isfinite(d)])
print(f"超声乳腺肿块数据集: 平均Dice = {dice_mean_us:.4f}, 平均IoU = {iou_mean_us:.4f}, 平均Hausdorff距离 = {hd_mean_us:.2f} pixel")


可视化一个超声图像的分割结果：

In [ ]:
# 随机选择一个超声图像进行可视化
idx = np.random.randint(0, len(ultrasound_images))
test_img = ultrasound_images[idx]
test_mask = ultrasound_masks[idx]

predictor.set_image(test_img)
ys, xs = np.where(test_mask == 1)
y_min, y_max = ys.min(), ys.max()
x_min, x_max = xs.min(), xs.max()
input_box = np.array([x_min, y_min, x_max, y_max])
masks, _, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
pred_mask = masks[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(test_img)
axes[0].imshow(np.ma.masked_where(test_mask==0, test_mask), cmap='cool', alpha=0.5)
axes[0].set_title("真值分割 (肿块)")
axes[0].axis('off')

axes[1].imshow(test_img)
axes[1].imshow(np.ma.masked_where(pred_mask==0, pred_mask), cmap='cool', alpha=0.5)
axes[1].set_title("MedSAM预测分割")
axes[1].axis('off')
plt.show()


左图显示乳腺超声图像上的肿块真值区域（蓝色遮罩），右图为MedSAM的预测结果。可以观察到，对于边界模糊、形状不规则的超声肿块，MedSAM的预测可能存在欠割或过割的情况。
分析与结果

我们汇总各数据集的平均评估指标如下：
Hippocampus MRI – Dice: {dice_mean_hc:.3f}, IoU: {iou_mean_hc:.3f}, Hausdorff距离: {hd_mean_hc:.1f}像素。
Spleen CT – Dice: {dice_mean_spleen:.3f}, IoU: {iou_mean_spleen:.3f}, Hausdorff距离: {hd_mean_spleen:.1f}像素。
Breast Ultrasound – Dice: {dice_mean_us:.3f}, IoU: {iou_mean_us:.3f}, Hausdorff距离: {hd_mean_us:.1f}像素。
下面的表格总结了各模态数据集上MedSAM原生模型的分割性能：